# MPPP — 02 COLMAP alignment (Navcam, optionally + Mastcam-Z)

The images processed in a WORK folder (notebook 01 or `mppp process`) in, a prior-aligned COLMAP block out
(`mppp.pipeline.align`; from the shell: `mppp align WORK`): one COLMAP camera per eye from the promoted models
(`mppp.calib`), SIFT at native resolution inside the masks, matching, a triangulation from the rover poses and a
weighted bundle adjustment with position and attitude priors (frames that do not align with the majority are
excluded; with Mastcam-Z the Navcam block is solved first and held), the health report with a top-down station map,
and the `error_input/` folder for notebook 03.

**Frame.** The block lives in East-North-Up metres of the landing site frame minus a rounded offset (`project.json`,
`error_input/summary.json`).  **Reruns** reuse the features while the images are the same and the matches while the
features and matching settings are the same; leave images out with `<WORK>/exclude_images.txt` (stations, sols,
sequences or file-name patterns), and put other settings side by side with `variant="name"` (-> `WORK/colmap_name`).

This kernel needs pip `pycolmap` 4.2 and `pyceres` 2.6 (`pip install -e .[sfm]`); GPU extraction and matching run
in `gpu_python` (`MPPP_GPU_PYTHON`).

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")   # Windows/conda: duplicate Intel OpenMP guard; before torch
import json
import logging
from pathlib import Path
import mppp
from mppp.pipeline import Settings
logging.basicConfig(level=logging.INFO, format="%(message)s")   # the pipeline's progress lines
print("MPPP", mppp.__version__, "from", Path(mppp.__file__).parent)
import pycolmap, pyceres
from mppp.sfm import check_ba_environment
print("pycolmap", pycolmap.__version__, "(CUDA:", pycolmap.has_cuda, ") | pyceres", pyceres.__version__)
check_ba_environment()        # stops here if this kernel cannot run the weighted BA (e.g. it is the conda CUDA env)

## 1  Parameters
Every setting is a field of `mppp.pipeline.Settings` (`help(Settings)`); `<WORK>/mppp_settings.json` and the site's `settings` in `sites.json` are read by `load_settings`.

In [ ]:
WORK = Path(os.environ.get("MPPP_SCAPES", "/scapes")) / "mars2020_sol_0460_rockytop"   # the WORK folder (holds processed/)
from mppp.pipeline import load_settings
settings = load_settings(WORK, base=Settings(
    variant="",                           # "" -> WORK/colmap; "name" -> WORK/colmap_name (other settings side by side)
    exclude=[],                           # images to leave out: "S032D1184", "sol:658", "seq:NCAM08111", "ZR0_0690_*"
    gui_files=False,                      # True: write colmap_gui.ini and open_in_colmap.bat into the project
    reprocess_all=False,                  # True: rebuild the project and redo the features
    # navcam_intrinsics="hold",           # keep the Navcam cameras at the promoted models
    # attitude_prior_deg=1.0,             # tighter attitude priors
))
STOP_AFTER = None                         # run only part of the pipeline: "project", "features", "database", "match", "reconstruct", "health"

## 2  Align
Project, features, database, matching, reconstruction, health report, export (sections 3-8 of the pipeline).

In [ ]:
from mppp.pipeline import align
report = align(WORK, settings, stop_after=STOP_AFTER)
print("run key", report.get("run_key"), "| verdict", report.get("verdict"), "| project", report.get("project"))

## 3  Alignment health — check before any analysis
Each check has provisional warn and fail thresholds (set a priori, not validated limits). The report is in
`<project>/health/` (`health.json`, `health.md`, `health.png`, `station_map.png`: a top-down view of the stations with
the prior-to-refined shift of every camera).

In [ ]:
from IPython.display import Image, display
from mppp.pipeline import load_results
from mppp.sfm import health_table
proj, rec, report, summary = load_results(WORK, settings.variant)      # from disk: also after a kernel restart
print(health_table(report))
files = report.get("files") or {}
for key in ("png", "station_map"):
    f = (Path(proj.root) / "health" / {"png": "health.png", "station_map": "station_map.png"}[key])
    if f.is_file():
        display(Image(filename=str(f)))
if report["verdict"] == "fail":
    print("\nFAIL: fix the alignment (matching, priors, masks, thresholds) before any error analysis.")

## 4  Cameras: start -> refined
The Navcam eyes and their temperature bins (Mastcam-Z focus bins: section 5).

In [ ]:
from mppp.sfm.reconstruction import camera_changes, convergence_statistics
for r in camera_changes(rec, proj):
    if r["missing"]:
        print(r["camera"], ": not in the reconstruction"); continue
    print(f"{r['camera']} ({r['images']} images){'  [temperature bin]' if r['thermal_bin'] else ''}: "
          + " ".join(f"{n} {a:.4g}->{b:.4g}" for n, a, b in zip(r["names"][:10], r["start"], r["refined"])))
conv = proj.settings["reconstruction"].get("convergence") or convergence_statistics(rec, proj)
print("tie points by convergence angle", conv["bins_deg"], "deg:", conv["points"], "| cross-station:", conv["cross_station_points"])
eps = (summary or {}).get("eps", {})
print(f"eps (rms residual / sqrt 2): same-station {eps.get('eps_intra_px', float('nan')):.3f} px, "
      f"cross-station {eps.get('eps_cross_px', float('nan')):.3f} px")

## 5  Mastcam-Z focus breathing
Written to `<project>/health/zcam_focus_breathing.{png,csv,json}` by the pipeline when the block has focus-bin cameras.

In [ ]:
from mppp.sfm import focus_breathing_table
if any(c.get("group") for c in proj.cameras.values()):
    for row in focus_breathing_table(proj, rec)[:30]:
        print(row)
    f = Path(proj.root) / "health" / "zcam_focus_breathing.png"
    if f.is_file():
        display(Image(filename=str(f)))
else:
    print("no Mastcam-Z focus-bin cameras")

## 6  Camera shifts and the export for the error analysis
`error_input/` (the native-resolution text model, `poses.csv`, `summary.json`) is what notebook 03 reads.

In [ ]:
import csv
import matplotlib.pyplot as plt
from mppp.sfm import plot_camera_shifts
with (Path(proj.root) / "error_input" / "poses.csv").open(encoding="utf-8") as f:
    pose_rows = list(csv.DictReader(f))
fig = plot_camera_shifts(proj, rows=pose_rows)
plt.show()
print("error_input:", Path(proj.root) / "error_input", "| run_done.json:", json.dumps(report.get("run_done") or {}, default=str)[:300])